In [54]:
BATCH_SIZE = 16
LABEL= "Friendly"

In [64]:
%load_ext autoreload
%autoreload 2
import os
from hireverse.utils.dataset_handler import DatasetHandler
from hireverse.utils.utils import BASE_DIR

participant_ids = DatasetHandler.get_participant_ids()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [88]:
import gc
import cv2
import numpy as np

def participant_frames_generator(participant_id, number_of_frames_in_batch=16):
    label = DatasetHandler.get_labels_dict(participant_id)[LABEL]
    total_frames = DatasetHandler.get_number_of_frames(participant_id)
    frame_yielder = DatasetHandler.yield_sorted_participant_frames_images(participant_id, is_image_greyscale=True)
    
    for j in range(0, total_frames, number_of_frames_in_batch):
        frames_batch = []
        for i in range(number_of_frames_in_batch):
            try:
                frame = next(frame_yielder)
                frame = frame.astype('float32') / 255.0  # Normalize
                frames_batch.append(frame)
            except StopIteration:
                break
        
        # Yield only if we have at least 1 frame
        if frames_batch:
            yield frames_batch, label


def to_100class(score_1_to_10):
    return np.floor((score_1_to_10 - 1) * 10 + np.random.uniform(0, 10))


def train_generator(train_ids, number_of_videos_in_the_batch=6):
    # Initialize the frame generators for each participant dynamically
    participants_frame_gens = {participant_id: participant_frames_generator(participant_id) for participant_id in train_ids}
    
    while True:  # Loop indefinitely for continuous training
        selected_ids = np.random.choice(train_ids, size=number_of_videos_in_the_batch, replace=False)
        X_train = []
        y_train = []
        
        for participant_id in selected_ids:
            participant_frame_gen = participants_frame_gens[participant_id]
            try:
                frames_batch, label = next(participant_frame_gen)
                X_train.append(frames_batch)
                y_train.append(to_100class(label))
            except StopIteration:
                # Skip exhausted generators (move to next participant)
                continue
        
        # Yield the batch only if we have enough data
        if X_train:
            yield np.array(X_train), np.array(y_train)

In [89]:
from sklearn.model_selection import train_test_split

# TODO: use group split
train_ids, temp_ids = train_test_split(participant_ids, test_size=0.5, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

train_gen = train_generator(train_ids)

In [90]:
X, y = next(train_gen)
print(X.shape)
print(y.shape)

(6, 16, 640, 640)
(6,)


In [91]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_competency_cnn(input_shape=(640, 640, 1), num_classes=600):
    model = models.Sequential()

    # Initial Convolutional Blocks
    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.25))

    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.25))

    # Final Convolutional Layers
    model.add(layers.Conv2D(2048, (3, 3), activation='relu'))
    model.add(layers.GlobalAveragePooling2D())

    # Final Classification Layer
    model.add(layers.Dense(num_classes, activation='softmax'))  # 6 competencies × 100 classes

    # Compile with learning rate (as per paper)
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    return model

# Initialize model
model = build_competency_cnn()
model.summary()

/Users/bassel27/personal_projects/hireverse/venv/lib/python3.9/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 638, 638, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 319, 319, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 319, 319, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 319, 319, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 317, 317, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 158, 158, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 158, 158, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 158, 158, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 156, 156, 2048) │     1,181,696 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 600)            │     1,229,400 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,430,296 (9.27 MB)

 Trainable params: 2,430,104 (9.27 MB)

 Non-trainable params: 192 (768.00 B)

In [92]:
model.fit(
    next(train_gen),
    epochs=10
)

Epoch 1/10


/Users/bassel27/personal_projects/hireverse/venv/lib/python3.9/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor_201
Received: inputs=('Tensor(shape=(None, 16, 640, 640))', 'Tensor(shape=(None,))')
  warnings.warn(msg)


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "conv2d_9" is incompatible with the layer: expected axis -1 of input shape to have value 1, but received input with shape (None, 16, 640, 640)[0m

Arguments received by Sequential.call():
  • inputs=('tf.Tensor(shape=(None, 16, 640, 640), dtype=float32)', 'tf.Tensor(shape=(None,), dtype=float32)')
  • training=True
  • mask=('None', 'None')